# LeetCode 1663 — Smallest String With A Given Numeric Value bằng Python

Link đề: https://leetcode.com/problems/smallest-string-with-a-given-numeric-value/ · Độ khó: **Medium** · Chủ đề: String, Greedy

Skeleton LeetCode:

```python
class Solution:
    def getSmallestString(self, n: int, k: int) -> str:
        ...
```

Notebook đi từ **duyệt mọi xâu theo thứ tự từ điển** (vét cạn) → **tham lam từ trái sang phải** ("đặt chữ nhỏ nhất còn khả thi") → **tham lam từ phải sang trái** ("đổ chữ `z` từ cuối") → **công thức đóng** dựng xâu bằng phép nhân chuỗi.

## 1. Mục tiêu học

1. Hiểu giá trị số của xâu (`a = 1, …, z = 26`) và thứ tự **từ điển**: chữ ở vị trí **đầu tiên khác nhau** quyết định.
2. Nắm kỹ thuật tham lam "**chọn nhỏ nhất ở vị trí đầu mà phần còn lại vẫn khả thi**" và điều kiện khả thi `n_còn ≤ k_còn ≤ 26 · n_còn`.
3. Nhận ra cấu trúc của đáp án: `a…a` + **một chữ giữa** + `z…z`, và dựng nó bằng công thức `O(n)` gọn.
4. Biết tránh các bẫy: **lệch một** khi đổi số ↔ chữ, ghép chuỗi `O(n²)`, công thức sai khi phần dư bằng 0.
5. Đối chiếu nhiều cách với vét cạn (với `n` nhỏ) và kiểm tra tính chất đáp án trên dữ liệu ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Giá trị số** của một chữ cái in thường là vị trí của nó trong bảng chữ cái (đánh số từ 1): `a = 1`, `b = 2`, `c = 3`, …, `z = 26`. Giá trị số của một xâu là **tổng** giá trị các chữ. Ví dụ `"abe"` có giá trị `1 + 2 + 5 = 8`.

Cho hai số nguyên `n` và `k`. Trả về xâu **nhỏ nhất theo thứ tự từ điển** có độ dài `n` và giá trị số **bằng** `k`.

(Xâu `x` nhỏ hơn `y` theo thứ tự từ điển nếu `x` là tiền tố của `y`, hoặc tại vị trí đầu tiên `i` mà `x[i] ≠ y[i]` thì `x[i]` đứng trước `y[i]` trong bảng chữ cái.)

**Ví dụ 1:** `n = 3`, `k = 27` → `"aay"` (`1 + 1 + 25 = 27`).

**Ví dụ 2:** `n = 5`, `k = 73` → `"aaszz"` (`1 + 1 + 19 + 26 + 26 = 73`).

**Ràng buộc:**
- `1 ≤ n ≤ 10⁵`
- `n ≤ k ≤ 26 · n` (luôn có nghiệm)

In [1]:
import random
import time
from itertools import product
from typing import List

examples = [(3, 27, "aay"), (5, 73, "aaszz")]

def value(s):
    return sum(ord(c) - ord('a') + 1 for c in s)

for n, k, exp in examples:
    print(f"n={n}, k={k} -> kỳ vọng {exp!r}: độ dài {len(exp)}, giá trị {value(exp)}")
print("giá trị 'abe' =", value("abe"))

n=3, k=27 -> kỳ vọng 'aay': độ dài 3, giá trị 27
n=5, k=73 -> kỳ vọng 'aaszz': độ dài 5, giá trị 73
giá trị 'abe' = 8


## 3. Kiến thức nền

### 3.1 `ord`, `chr` và đổi giữa chữ ↔ giá trị

`ord('a') = 97`. Giá trị của chữ `c` là `ord(c) − 96`; chữ có giá trị `v` là `chr(v + 96)` (tức `chr(ord('a') + v − 1)`). Dễ lệch một ở đây!

In [2]:
print([(c, ord(c) - 96) for c in "abcyz"])
print([chr(v + 96) for v in (1, 2, 25, 26)])
print("lệch một nếu dùng chr(ord('a') + v):", [chr(ord('a') + v) for v in (1, 26)])

[('a', 1), ('b', 2), ('c', 3), ('y', 25), ('z', 26)]
['a', 'b', 'y', 'z']
lệch một nếu dùng chr(ord('a') + v): ['b', '{']


### 3.2 So sánh xâu theo thứ tự từ điển

Python so sánh xâu đúng theo thứ tự từ điển. Chữ **đầu tiên** khác nhau quyết định — nên muốn xâu nhỏ, phải làm **phần đầu** nhỏ nhất có thể.

In [3]:
print("aay" < "aaz", "aay" < "abx", "azz" < "baa")
print(sorted(["aaz", "aby", "aay", "bax"]))

True True True
['aay', 'aaz', 'aby', 'bax']


### 3.3 Xây xâu hiệu quả: list + `''.join` và phép nhân chuỗi

Ghép chuỗi lặp lại kiểu `s = c + s` tốn `O(len(s))` mỗi lần ⇒ `O(n²)`. Hãy gom vào **list** rồi `''.join`, hoặc dùng **phép nhân chuỗi** `'a' * m` (chạy ở tốc độ C).

In [4]:
parts = ['a'] * 3
parts[-1] = 'y'
print(''.join(parts), "| nhân chuỗi:", 'a' * 2 + 's' + 'z' * 2)

aay | nhân chuỗi: aaszz


### 3.4 `divmod`

`divmod(a, b)` trả `(a // b, a % b)` trong một lần gọi — tiện khi đếm "bao nhiêu chữ `z`" và "phần dư".

In [5]:
print(divmod(73 - 5, 25), "-> 2 lần +25 và dư 18")

(2, 18) -> 2 lần +25 và dư 18


## 4. Ý tưởng nền tảng / trực giác

Khởi đầu từ xâu nhỏ nhất có thể: `"aaa…a"` (giá trị `n`). Ta còn **thừa** `d = k − n` đơn vị phải "đổ" vào. Mỗi vị trí nhận tối đa `25` (từ `a` lên `z`).

Để xâu **nhỏ nhất theo từ điển**, các vị trí **đầu** phải giữ chữ `a` lâu nhất có thể ⇒ đổ phần thừa vào các vị trí **cuối** trước, mỗi vị trí đổ đầy `25` (thành `z`), vị trí cuối cùng nhận phần lẻ.

Tính tay ví dụ 2: `n = 5`, `k = 73` ⇒ `d = 68`.
- Vị trí 4: đổ 25 ⇒ `z`, còn 43.
- Vị trí 3: đổ 25 ⇒ `z`, còn 18.
- Vị trí 2: đổ 18 ⇒ `a + 18 = s`.
- Vị trí 0, 1: giữ `a`. ⇒ `"aaszz"`.

Cách nhìn **từ trái sang phải**: ở vị trí `i`, chọn chữ nhỏ nhất `v` sao cho phần còn lại (`n − i − 1` vị trí) vẫn đủ chỗ chứa `k − v` (tối đa `26` mỗi vị trí): `v = max(1, k − 26·(n − i − 1))`.

In [6]:
n, k = 5, 73
d = k - n
s = ['a'] * n
i = n - 1
while d > 0:
    add = min(25, d)
    s[i] = chr(ord('a') + add)
    print(f"vị trí {i}: đổ {add} -> '{s[i]}', còn thừa {d - add}, xâu = {''.join(s)}")
    d -= add
    i -= 1

vị trí 4: đổ 25 -> 'z', còn thừa 43, xâu = aaaaz
vị trí 3: đổ 25 -> 'z', còn thừa 18, xâu = aaazz
vị trí 2: đổ 18 -> 's', còn thừa 0, xâu = aaszz


Một ví dụ nhỏ để "thấy" thứ tự từ điển: liệt kê **mọi** xâu độ dài 2 có giá trị 27, đã sắp xếp. Xâu nhỏ nhất là `"az"` — chữ đầu `a` nhỏ nhất có thể vì chữ sau vẫn chứa được `26`. Xâu độ dài 3 giá trị 27 thì có rất nhiều; đứng đầu là `"aay"` (ví dụ 1).

In [7]:
pairs = sorted(a + b for a in "abcdefghijklmnopqrstuvwxyz" for b in "abcdefghijklmnopqrstuvwxyz" if value(a + b) == 27)
print(len(pairs), "xâu độ dài 2 có giá trị 27:", pairs)
triples = sorted(a + b + c for a in "abcdefghijklmnopqrstuvwxyz" for b in "abcdefghijklmnopqrstuvwxyz" for c in "abcdefghijklmnopqrstuvwxyz" if value(a + b + c) == 27)
print(len(triples), "xâu độ dài 3 có giá trị 27; 5 xâu đầu:", triples[:5], "| 3 xâu cuối:", triples[-3:])

26 xâu độ dài 2 có giá trị 27: ['az', 'by', 'cx', 'dw', 'ev', 'fu', 'gt', 'hs', 'ir', 'jq', 'kp', 'lo', 'mn', 'nm', 'ol', 'pk', 'qj', 'ri', 'sh', 'tg', 'uf', 've', 'wd', 'xc', 'yb', 'za']
325 xâu độ dài 3 có giá trị 27; 5 xâu đầu: ['aay', 'abx', 'acw', 'adv', 'aeu'] | 3 xâu cuối: ['xab', 'xba', 'yaa']


Nhận xét: trong danh sách độ dài 3, các xâu bắt đầu bằng `a` đứng trước, và trong số đó các xâu `aa?` đứng trước — đúng như tham lam: giữ chữ đầu nhỏ nhất **miễn là phần sau còn khả thi** (`"aa?"` cần `? = 25 = y`, vẫn ≤ 26 nên được).

## 5. Cách 1 — Brute Force / baseline: duyệt xâu theo thứ tự từ điển

**Ý tưởng:** `itertools.product("abc…z", repeat=n)` sinh các xâu **đúng theo thứ tự từ điển**. Xâu đầu tiên có giá trị `k` chính là đáp án.

In [8]:
ALPHA = "abcdefghijklmnopqrstuvwxyz"

def smallest_brute(n, k):
    for tup in product(ALPHA, repeat=n):
        if value(tup) == k:
            return ''.join(tup)
    return None

for n, k, exp in examples[:1]:
    t0 = time.perf_counter()
    print(smallest_brute(n, k), "(kỳ vọng", exp, ")", f"{(time.perf_counter() - t0) * 1000:.1f} ms")
print("n=2, k=52 ->", smallest_brute(2, 52))

aay (kỳ vọng aay ) 0.0 ms
n=2, k=52 -> zz


**Độ phức tạp:** xấu nhất duyệt `26ⁿ` xâu, mỗi xâu tính tổng `O(n)` ⇒ `O(26ⁿ · n)`. `n = 5` đã là `1.2·10⁷` xâu. Không khả thi với `n = 10⁵`.

**Nút thắt:** ta thử cả những tiền tố **chắc chắn vô vọng** (ví dụ `"aa…"` khi `k` quá lớn so với số chỗ còn lại). Cần một tiêu chí để biết **ngay** một tiền tố có hoàn thành được hay không.

## 6. Ý tưởng thuật toán / bước đột phá

**Điều kiện khả thi:** với `m` vị trí còn trống và tổng còn thiếu `r`, hoàn thành được ⇔ `m ≤ r ≤ 26 m` (mỗi chữ trong `[1, 26]`).

**Tham lam trái → phải:** ở vị trí `i` (còn `m = n − i − 1` vị trí sau nó), chọn chữ **nhỏ nhất** `v ∈ [1, 26]` sao cho `k − v` khả thi với `m` vị trí:
- `k − v ≤ 26 m` ⇔ `v ≥ k − 26 m`;
- `k − v ≥ m` ⇔ `v ≤ k − m` (luôn thỏa với `v` nhỏ nhất nếu bài có nghiệm).
⇒ `v = max(1, k − 26 m)`. **Nếu** `k − 26m ≤ 1` **thì** đặt `a`; **ngược lại** đặt chữ có giá trị `k − 26 m` (khi đó mọi vị trí sau đều phải là `z`).

**Hệ quả cấu trúc:** đáp án luôn có dạng `a^p` + `c` + `z^q` (có thể không có chữ giữa, hoặc chữ giữa là `a`/`z`). Số chữ `z` là `q = (k − n) // 25`, phần dư `r = (k − n) % 25` cộng vào chữ giữa.

**Bất biến (trái → phải):** trước khi xử lý vị trí `i`, tổng còn thiếu `k_i` thỏa `n − i ≤ k_i ≤ 26 (n − i)`, và tiền tố đã đặt là tiền tố nhỏ nhất có thể của mọi đáp án hợp lệ.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam từ trái sang phải

In [9]:
def smallest_left_to_right(n, k):
    out = []
    for i in range(n):
        m = n - i - 1                     # số vị trí còn lại sau i
        v = max(1, k - 26 * m)            # chữ nhỏ nhất còn khả thi
        out.append(chr(96 + v))
        k -= v
    return ''.join(out)

class SolutionLTR:
    def getSmallestString(self, n: int, k: int) -> str:
        out = []
        for i in range(n):
            v = max(1, k - 26 * (n - i - 1))
            out.append(chr(96 + v))
            k -= v
        return ''.join(out)

for n, k, exp in examples:
    print(smallest_left_to_right(n, k), SolutionLTR().getSmallestString(n, k), "(kỳ vọng", exp, ")")

aay aay (kỳ vọng aay )
aaszz aaszz (kỳ vọng aaszz )


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ cho kết quả.

### 7.2 Cách 3 — Tham lam từ phải sang trái ("đổ `z` từ cuối")

Bắt đầu với `['a'] * n`, phần thừa `d = k − n`. Từ vị trí cuối: **nếu** `d > 25` **thì** đặt `z`, `d −= 25`, lùi sang trái; **ngược lại** cộng `d` vào chữ hiện tại và dừng.

In [10]:
def smallest_right_to_left(n, k):
    ans = ['a'] * n
    i, d = n - 1, k - n
    while d > 25:
        ans[i] = 'z'
        d -= 25
        i -= 1
    if d > 0:
        ans[i] = chr(ord('a') + d)
    return ''.join(ans)

for n, k, exp in examples:
    print(smallest_right_to_left(n, k), "(kỳ vọng", exp, ")")
print("biên: n=1,k=1 ->", smallest_right_to_left(1, 1), "| n=1,k=26 ->", smallest_right_to_left(1, 26), "| n=3,k=78 ->", smallest_right_to_left(3, 78))

aay (kỳ vọng aay )
aaszz (kỳ vọng aaszz )
biên: n=1,k=1 -> a | n=1,k=26 -> z | n=3,k=78 -> zzz


Khi `d ≤ 25` thoát vòng lặp, `i ≥ 0` luôn đúng vì `k ≤ 26n` ⇒ `d ≤ 25n`. Nếu `d == 0` thì giữ `a` (dòng `if d > 0` chỉ để rõ ràng — cộng 0 cũng ra `a`). **Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.3 Cách 4 — Công thức đóng bằng phép nhân chuỗi

`q, r = divmod(k − n, 25)`:
- `q` chữ `z` ở cuối;
- **nếu** `q == n` (chỉ xảy ra khi `k = 26n`) **thì** đáp án là `'z' * n`;
- **ngược lại**: `'a' * (n − q − 1)` + `chr(97 + r)` + `'z' * q`.

In [11]:
def smallest_formula(n, k):
    q, r = divmod(k - n, 25)
    if q == n:
        return 'z' * n
    return 'a' * (n - q - 1) + chr(ord('a') + r) + 'z' * q

for n, k, exp in examples:
    print(smallest_formula(n, k), "(kỳ vọng", exp, ")")
print("n=2, k=52 ->", smallest_formula(2, 52), "| n=4, k=29 ->", smallest_formula(4, 29))

aay (kỳ vọng aay )
aaszz (kỳ vọng aaszz )
n=2, k=52 -> zz | n=4, k=29 -> aaaz


**Độ phức tạp:** `O(n)` (để tạo xâu kết quả), nhưng nhanh hơn hẳn trong Python vì phép nhân chuỗi chạy bằng C. Đây là lời giải chính (mục 9).

## 8. TRACE từng bước

### 8.1 Trace cách 2 (trái → phải) trên ví dụ 2: `n = 5`, `k = 73`

| i | m = n−i−1 | k trước | v = max(1, k − 26m) | chữ | k sau |
|---|---|---|---|---|---|
| 0 | 4 | 73 | max(1, 73−104) = 1 | a | 72 |
| 1 | 3 | 72 | max(1, 72−78) = 1 | a | 71 |
| 2 | 2 | 71 | max(1, 71−52) = 19 | s | 52 |
| 3 | 1 | 52 | max(1, 52−26) = 26 | z | 26 |
| 4 | 0 | 26 | 26 | z | 0 |

In [12]:
def trace_ltr(n, k):
    out = []
    for i in range(n):
        m = n - i - 1
        v = max(1, k - 26 * m)
        print(f"i={i}: m={m}, k={k:>3}, v=max(1, {k}-{26 * m})={v} -> '{chr(96 + v)}', còn k={k - v}  | khả thi: {m} <= {k - v} <= {26 * m}")
        out.append(chr(96 + v)); k -= v
    print("=>", ''.join(out))
    return ''.join(out)

trace_ltr(5, 73)

i=0: m=4, k= 73, v=max(1, 73-104)=1 -> 'a', còn k=72  | khả thi: 4 <= 72 <= 104
i=1: m=3, k= 72, v=max(1, 72-78)=1 -> 'a', còn k=71  | khả thi: 3 <= 71 <= 78
i=2: m=2, k= 71, v=max(1, 71-52)=19 -> 's', còn k=52  | khả thi: 2 <= 52 <= 52
i=3: m=1, k= 52, v=max(1, 52-26)=26 -> 'z', còn k=26  | khả thi: 1 <= 26 <= 26
i=4: m=0, k= 26, v=max(1, 26-0)=26 -> 'z', còn k=0  | khả thi: 0 <= 0 <= 0
=> aaszz


'aaszz'

### 8.2 Trace cách 3 (phải → trái) trên ví dụ 1: `n = 3`, `k = 27`

In [13]:
def trace_rtl(n, k):
    ans = ['a'] * n
    i, d = n - 1, k - n
    print(f"bắt đầu: {''.join(ans)}, thừa d={d}")
    while d > 25:
        ans[i] = 'z'; d -= 25
        print(f"  i={i}: đặt 'z', còn d={d} -> {''.join(ans)}")
        i -= 1
    ans[i] = chr(ord('a') + d)
    print(f"  i={i}: cộng {d} -> '{ans[i]}' -> {''.join(ans)}")
    return ''.join(ans)

trace_rtl(3, 27)
trace_rtl(6, 100)

bắt đầu: aaa, thừa d=24
  i=2: cộng 24 -> 'y' -> aay
bắt đầu: aaaaaa, thừa d=94
  i=5: đặt 'z', còn d=69 -> aaaaaz
  i=4: đặt 'z', còn d=44 -> aaaazz
  i=3: đặt 'z', còn d=19 -> aaazzz
  i=2: cộng 19 -> 't' -> aatzzz


'aatzzz'

### 8.3 Công thức trên vài bộ `(n, k)` đặc biệt

In [14]:
for n, k in [(1, 1), (1, 26), (3, 3), (3, 78), (4, 29), (4, 28), (5, 73), (10, 100)]:
    q, r = divmod(k - n, 25)
    print(f"n={n:>2}, k={k:>3}: k-n={k - n:>3}, q={q}, r={r:>2} -> {smallest_formula(n, k)!r}")

n= 1, k=  1: k-n=  0, q=0, r= 0 -> 'a'
n= 1, k= 26: k-n= 25, q=1, r= 0 -> 'z'
n= 3, k=  3: k-n=  0, q=0, r= 0 -> 'aaa'
n= 3, k= 78: k-n= 75, q=3, r= 0 -> 'zzz'
n= 4, k= 29: k-n= 25, q=1, r= 0 -> 'aaaz'
n= 4, k= 28: k-n= 24, q=0, r=24 -> 'aaay'
n= 5, k= 73: k-n= 68, q=2, r=18 -> 'aaszz'
n=10, k=100: k-n= 90, q=3, r=15 -> 'aaaaaapzzz'


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def getSmallestString(self, n: int, k: int) -> str:
        q, r = divmod(k - n, 25)          # q chữ 'z', phần dư r cộng vào chữ giữa
        if q == n:
            return 'z' * n
        return 'a' * (n - q - 1) + chr(ord('a') + r) + 'z' * q

for n, k, exp in examples:
    assert Solution().getSmallestString(n, k) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Tham lam trái → phải (cách 2).** Giả sử bất biến đúng trước vị trí `i`: tổng còn thiếu `k_i` khả thi với `n − i` vị trí. Mọi đáp án hợp lệ có cùng tiền tố đã đặt phải chọn ở vị trí `i` một chữ `v` sao cho phần sau khả thi, tức `v ≥ k_i − 26 m` và `v ≥ 1`. Chọn `v* = max(1, k_i − 26 m)` là **nhỏ nhất có thể** ⇒ bất kỳ xâu hợp lệ nào cũng có chữ ở vị trí `i` `≥ v*` ⇒ xâu của ta không lớn hơn (theo từ điển, vị trí đầu tiên khác nhau quyết định). Kiểm tra bất biến cho bước sau: `k_i − v* ≤ 26m` (theo cách chọn) và `k_i − v* ≥ m` (vì `k_i ≥ (m + 1)` và nếu `v* = k_i − 26 m > 1` thì `k_i − v* = 26m ≥ m`; nếu `v* = 1` thì `k_i − 1 ≥ m`).

**Cấu trúc đáp án.** Theo công thức `v = max(1, k − 26 m)`: khi `v > 1` thì phần còn lại đúng bằng `26 m` ⇒ toàn `z`. Vậy xâu có dạng `a…a` (các vị trí `v = 1`), rồi **một** chữ `v > 1` (có thể không có), rồi toàn `z`. Số chữ `z` cuối `q` và chữ giữa xác định bởi `k − n = 25 q + r` với `0 ≤ r < 25` (chữ giữa là `a + r`); trường hợp `q = n` là xâu toàn `z`. Đó đúng là cách 3 và cách 4.

**Có nghiệm** vì `n ≤ k ≤ 26n` (đề bảo đảm).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Duyệt `26ⁿ` xâu | `O(26ⁿ · n)` | `O(n)` |
| 2. Tham lam trái → phải | `O(n)` | `O(n)` |
| 3. Tham lam phải → trái | `O(n)` | `O(n)` |
| 4. Công thức nhân chuỗi | `O(n)` (hằng số rất nhỏ) | `O(n)` |

Kết quả là xâu độ dài `n` nên không thể nhanh hơn `O(n)`. Đo trên `n = 10⁵`, và so với cách ghép chuỗi `O(n²)` (chỉ với `n = 20000`):

In [16]:
n, k = 10**5, 1_300_000
for name, fn in [("ltr", smallest_left_to_right), ("rtl", smallest_right_to_left), ("formula", smallest_formula), ("Solution", Solution().getSmallestString)]:
    t0 = time.perf_counter(); s = fn(n, k); dt = time.perf_counter() - t0
    print(f"{name:<9}: len={len(s)}, value={value(s)}, {dt * 1000:6.2f} ms")

def smallest_prepend(n, k):
    s = ""
    for i in range(n - 1, -1, -1):
        m = i                             # số vị trí còn lại phía trước
        v = min(26, k - m)                # chữ lớn nhất cho vị trí cuối còn khả thi
        s = chr(96 + v) + s               # ghép vào đầu: O(len(s)) mỗi lần
        k -= v
    return s

def smallest_list_rtl(n, k):
    out = []
    for i in range(n - 1, -1, -1):
        v = min(26, k - i); out.append(chr(96 + v)); k -= v
    return ''.join(reversed(out))

for N in (20000, 100000):
    t0 = time.perf_counter(); s2 = smallest_prepend(N, 13 * N); a = time.perf_counter() - t0
    t0 = time.perf_counter(); s3 = smallest_list_rtl(N, 13 * N); b = time.perf_counter() - t0
    assert s2 == s3 == smallest_formula(N, 13 * N)
    print(f"n={N:>6}: ghép vào đầu {a * 1000:7.1f} ms | list + join {b * 1000:6.1f} ms")

ltr      : len=100000, value=1300000,  10.49 ms
rtl      : len=100000, value=1300000,   2.17 ms
formula  : len=100000, value=1300000,   0.02 ms
Solution : len=100000, value=1300000,   0.01 ms
n= 20000: ghép vào đầu     3.6 ms | list + join    1.3 ms


n=100000: ghép vào đầu   126.1 ms | list + join    7.3 ms


## 12. Bộ test

1. Ví dụ đề.
2. Edge case: `k = n` (toàn `a`), `k = 26n` (toàn `z`), `n = 1`, `k − n` chia hết cho 25.
3. **Toàn bộ** bộ `(n, k)` với `n ≤ 3` so với vét cạn.
4. Ngẫu nhiên lớn: kiểm tra **tính chất** (độ dài, giá trị, dạng `a*cz*`) và so giữa các cách.
5. Kiểm tra "tối ưu cục bộ": không thể giảm một vị trí sớm hơn mà vẫn giữ tổng (so với đáp án tăng một chữ ở sau).

In [17]:
import re

ALL = {
    "ltr": smallest_left_to_right,
    "rtl": smallest_right_to_left,
    "formula": smallest_formula,
    "SolLTR": lambda n, k: SolutionLTR().getSmallestString(n, k),
    "Solution": lambda n, k: Solution().getSmallestString(n, k),
}
SHAPE = re.compile(r"^a*[a-z]?z*$")

def check(n, k, exp=None, brute=False):
    res = {name: fn(n, k) for name, fn in ALL.items()}
    s = res["Solution"]
    assert len(set(res.values())) == 1, (n, k, res)
    assert len(s) == n and value(s) == k and SHAPE.match(s), (n, k, s)
    if brute:
        assert s == smallest_brute(n, k), (n, k)
    if exp is not None:
        assert s == exp, (n, k, s, exp)
    return s

for n, k, exp in examples:
    check(n, k, exp)
print("Ví dụ đề: OK")

check(1, 1, "a"); check(1, 26, "z"); check(4, 4, "aaaa"); check(4, 104, "zzzz")
check(2, 27, "az"); check(3, 53, "azz"); check(3, 28, "aaz"); check(2, 52, "zz")
print("Edge case: OK")

cnt = 0
for n in range(1, 4):
    for k in range(n, 26 * n + 1):
        check(n, k, brute=True); cnt += 1
print(f"Toàn bộ {cnt} bộ (n, k) với n <= 3 so với vét cạn: OK")

Ví dụ đề: OK
Edge case: OK


Toàn bộ 153 bộ (n, k) với n <= 3 so với vét cạn: OK


In [18]:
rnd = random.Random(1663)
for _ in range(3000):
    n = rnd.randint(1, 200)
    k = rnd.randint(n, 26 * n)
    s = check(n, k)
    # tối ưu: thử giảm một chữ không phải 'a' ở vị trí i và tăng một chữ sau đó -> phải là không thể
    i = next((j for j, c in enumerate(s) if c != 'a'), None)
    if i is not None:
        assert all(c == 'z' for c in s[i + 1:]), "sau chữ giữa phải toàn 'z' nên không còn chỗ tăng"
print("3000 bộ ngẫu nhiên: OK")

for _ in range(20):
    n = rnd.randint(10**4, 10**5)
    check(n, rnd.randint(n, 26 * n))
print("20 bộ cỡ lớn: OK")

3000 bộ ngẫu nhiên: OK


20 bộ cỡ lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam ngược: đặt chữ lớn ở đầu

In [19]:
def smallest_bug_big_first(n, k):
    out = []
    for i in range(n):
        m = n - i - 1
        v = min(26, k - m)                 # SAI: chữ lớn nhất còn khả thi -> xâu LỚN nhất
        out.append(chr(96 + v)); k -= v
    return ''.join(out)

for n, k, exp in examples:
    print(f"n={n}, k={k}: sai = {smallest_bug_big_first(n, k)!r}, đúng = {exp!r}")

n=3, k=27: sai = 'yaa', đúng = 'aay'
n=5, k=73: sai = 'zzsaa', đúng = 'aaszz'


**Vì sao sai:** thứ tự từ điển xét **từ trái sang**; chữ lớn ở đầu làm xâu lớn. Bản sai trả về xâu **lớn nhất**. **Sửa:** chọn chữ **nhỏ nhất** còn khả thi: `max(1, k − 26 m)`.

### Lỗi 2 — Lệch một khi đổi giá trị ↔ chữ

In [20]:
def smallest_bug_offby1(n, k):
    out = []
    for i in range(n):
        v = max(1, k - 26 * (n - i - 1))
        out.append(chr(ord('a') + v))       # SAI: giá trị 1 phải là 'a', không phải 'b'
        k -= v
    return ''.join(out)

s = smallest_bug_offby1(3, 27)
print("sai :", repr(s), "giá trị", value(s) if all('a' <= c <= 'z' for c in s) else "(có ký tự ngoài a-z)")
print("đúng:", repr(Solution().getSmallestString(3, 27)))
print("với k = 26n:", repr(smallest_bug_offby1(2, 52)), "<- '{' không phải chữ cái")

sai : 'bbz' giá trị 30
đúng: 'aay'
với k = 26n: '{{' <- '{' không phải chữ cái


**Vì sao sai:** `chr(ord('a') + v)` cho chữ có giá trị `v + 1`. Với `v = 26` còn ra ký tự `'{'`. **Sửa:** `chr(96 + v)` hoặc `chr(ord('a') + v − 1)`. (Cách 3/4 dùng `chr(ord('a') + d)` là **đúng** vì ở đó `d` là **lượng cộng thêm** vào `a`, không phải giá trị.)

### Lỗi 3 — Công thức quên trường hợp `q == n`

In [21]:
def smallest_bug_formula(n, k):
    q, r = divmod(k - n, 25)
    return 'a' * (n - q - 1) + chr(ord('a') + r) + 'z' * q        # SAI khi q == n

for n, k in [(2, 52), (1, 26)]:
    s = smallest_bug_formula(n, k)
    print(f"n={n}, k={k}: sai = {s!r} (độ dài {len(s)}), đúng = {Solution().getSmallestString(n, k)!r}")

n=2, k=52: sai = 'azz' (độ dài 3), đúng = 'zz'
n=1, k=26: sai = 'az' (độ dài 2), đúng = 'z'


**Vì sao sai:** khi `k = 26n` thì `q = n`, `n − q − 1 = −1` ⇒ `'a' * (−1) = ''` nhưng vẫn thêm một chữ giữa `a` ⇒ xâu dài `n + 1`. **Sửa:** xử lý riêng `q == n`.

### Lỗi 4 — Ghép chuỗi `O(n²)` trong vòng lặp

Viết `s = chr(...) + s` (ghép vào đầu) trong vòng lặp `n` lần: mỗi lần tạo xâu mới dài dần ⇒ tổng `~n²/2` ký tự được sao chép, `O(n²)`. Mục 11 đã đo: `memcpy` của CPython rất nhanh nên với `n = 20000` chưa thấy rõ, nhưng khi `n` tăng 5 lần thì thời gian tăng khoảng **25 lần** (dấu hiệu bậc hai), trong khi bản list + `join` chỉ tăng ~5 lần. Với ràng buộc lớn hơn hoặc ngôn ngữ khác, đây là nguyên nhân TLE kinh điển. **Sửa:** dùng list + `''.join`, hoặc phép nhân chuỗi.

### Lỗi 5 — Vòng `while d > 0` đổ `min(25, d)` nhưng quên lùi chỉ số

In [22]:
def smallest_bug_no_move(n, k):
    ans = ['a'] * n
    i, d = n - 1, k - n
    steps = 0
    while d > 0 and steps < 10:            # giới hạn bước để demo không lặp mãi
        add = min(25, d)
        ans[i] = chr(ord('a') + add)       # SAI: ghi đè cùng một vị trí, quên i -= 1
        d -= add
        steps += 1
    return ''.join(ans)

s = smallest_bug_no_move(5, 73)
print("sai :", repr(s), "giá trị", value(s), "| đúng:", repr(Solution().getSmallestString(5, 73)))

sai : 'aaaas' giá trị 23 | đúng: 'aaszz'


**Vì sao sai:** mọi lần đổ đều ghi đè vị trí cuối ⇒ tổng mất phần đã đổ trước. **Sửa:** `i -= 1` sau mỗi lần đổ đầy.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xâu/số **nhỏ nhất theo thứ tự từ điển** thỏa ràng buộc tổng" ⇒ **tham lam từng vị trí từ trái sang**, chọn giá trị nhỏ nhất mà phần còn lại **vẫn khả thi**.
- Điều kiện khả thi với `m` chỗ, mỗi chỗ trong `[lo, hi]`: `m·lo ≤ còn_lại ≤ m·hi`.
- Muốn **lớn nhất** thì đối xứng: chọn giá trị lớn nhất còn khả thi.
- Đáp án tham lam kiểu này thường có dạng "min…min, một giá trị giữa, max…max" ⇒ có công thức đóng.
- Cẩn thận `chr/ord` lệch một; xây xâu bằng `''.join` hoặc nhân chuỗi.

## 15. Biến thể của bài

1. **Xâu lớn nhất theo từ điển** với cùng ràng buộc: `z…z` + chữ giữa + `a…a`.
2. **Bảng chữ cái giới hạn** (chỉ dùng `m` chữ đầu): thay `26` bằng `m`, `25` bằng `m − 1`.
3. **Số nhỏ nhất có `n` chữ số và tổng chữ số `k`** (chữ số đầu ≥ 1): cùng kỹ thuật với `[0, 9]` và ràng buộc chữ số đầu.
4. **LeetCode 1980 / 2160 / 1689…** các bài "dựng số/xâu tối ưu" bằng tham lam từng vị trí.
5. **Đếm số xâu** độ dài `n` có giá trị `k`: quy hoạch động / bao hàm–loại trừ (khác hẳn — không còn tham lam).

In [23]:
def largest_string(n, k):
    q, r = divmod(k - n, 25)
    if q == n:
        return 'z' * n
    return 'z' * q + chr(ord('a') + r) + 'a' * (n - q - 1)

def smallest_number_digit_sum(n, k):
    """Số nhỏ nhất có n chữ số (không có số 0 đứng đầu) với tổng chữ số k; None nếu không có."""
    if not (1 <= k <= 9 * n):
        return None
    out = []
    for i in range(n):
        m = n - i - 1
        lo = 1 if i == 0 else 0
        d = max(lo, k - 9 * m)
        out.append(str(d)); k -= d
    return ''.join(out)

print("lớn nhất n=5, k=73:", largest_string(5, 73), "| giá trị", value(largest_string(5, 73)))
print("số nhỏ nhất 3 chữ số, tổng 20:", smallest_number_digit_sum(3, 20), "| 2 chữ số, tổng 1:", smallest_number_digit_sum(2, 1))
assert smallest_number_digit_sum(3, 20) == min(str(x) for x in range(100, 1000) if sum(map(int, str(x))) == 20)

lớn nhất n=5, k=73: zzsaa | giá trị 73
số nhỏ nhất 3 chữ số, tổng 20: 299 | 2 chữ số, tổng 1: 10


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | `product` theo thứ tự từ điển | `O(26ⁿ n)` | `O(n)` | kiểm thử `n ≤ 3` |
| 2. Trái → phải | `v = max(1, k − 26m)` | `O(n)` | `O(n)` | khuôn tham lam tổng quát |
| 3. Phải → trái | đổ `z` từ cuối | `O(n)` | `O(n)` | trực quan |
| 4. Công thức | `a^p c z^q` bằng nhân chuỗi | `O(n)` | `O(n)` | **nộp bài**, nhanh nhất |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().getSmallestString(4, 30)` và `Solution().getSmallestString(3, 52)`?

**Bài 2 (tính tay):** Trace cách 2 cho `n = 6`, `k = 100`: lập bảng `i`, `m`, `k`, `v`.

**Bài 3 (điền TODO):** Hoàn thiện `smallest_with_alphabet(n, k, m)` khi chỉ được dùng `m` chữ cái đầu (`a` … chữ thứ `m`); trả `None` nếu vô nghiệm.

**Bài 4 (tự viết lại):** Viết lại cách 3 nhưng không dùng vòng `while` — tính trực tiếp chỉ số bắt đầu dãy `z` và gán bằng slicing trên list.

**Bài 5 (mini challenge):** Viết `kth_smallest_string(n, k, t)` trả về xâu **nhỏ thứ `t`** (1-indexed) theo thứ tự từ điển trong số các xâu độ dài `n` có giá trị `k` (`n ≤ 30`), dùng DP đếm `ways[m][s]` = số xâu độ dài `m` giá trị `s`. Kiểm với vét cạn cho `n ≤ 3`.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def smallest_with_alphabet(n, k, m):
    # TODO: khả thi khi n <= k <= m*n; v = max(1, k - m*(n-i-1))
    pass

# Bài 4 — TODO
def smallest_rtl_slicing(n, k):
    # TODO: q, r = divmod(k - n, 25); ans = ['a']*n; ans[n-q:] = ['z']*q; xử lý chữ giữa
    pass

# Bài 5 — TODO
def kth_smallest_string(n, k, t):
    # TODO: ways[m][s]; ở mỗi vị trí thử chữ từ 'a', trừ dần số xâu bắt đầu bằng chữ đó
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `k − n = 26` và `49`.
- Mức 2: `divmod(26, 25) = (1, 1)`, `divmod(49, 25) = (1, 24)`.
- Mức 3: `"aabz"` và `"ayz"`.

**Bài 2**
- Mức 1: `m` lần lượt `5, 4, 3, 2, 1, 0`.
- Mức 2: Các vị trí đầu cho `v = 1` cho tới khi `k − 26m > 1`.
- Mức 3: Kết quả có 3 chữ `z` ở cuối.

**Bài 3**
- Mức 1: Thay 26 bằng `m`.
- Mức 2: Kiểm tra `n ≤ k ≤ m n` trước.
- Mức 3: Chữ có giá trị `v` là `chr(96 + v)`.

**Bài 4**
- Mức 1: `q` chữ `z` nằm ở `ans[n − q:]`.
- Mức 2: Nếu `q < n`, chữ giữa ở `ans[n − q − 1]` được cộng `r`.
- Mức 3: Trường hợp `q == n` thì slicing gán toàn bộ.

**Bài 5**
- Mức 1: `ways[0][0] = 1`, `ways[m][s] = Σ_{v=1..26} ways[m−1][s−v]`.
- Mức 2: Ở vị trí `i` (còn `m` chỗ sau), thử `v = 1, 2, …`: nếu `t ≤ ways[m][k − v]` thì chọn `v`, ngược lại `t −= ways[m][k − v]`.
- Mức 3: `t = 1` phải cho đúng `Solution()`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
r1 = (Solution().getSmallestString(4, 30), Solution().getSmallestString(3, 52))
print("Bài 1:", r1)
assert r1 == ("aabz", "ayz")

# Bài 2
assert trace_ltr(6, 100) == Solution().getSmallestString(6, 100)

# Bài 3
def smallest_with_alphabet_sol(n, k, m):
    if not (n <= k <= m * n):
        return None
    out = []
    for i in range(n):
        v = max(1, k - m * (n - i - 1))
        out.append(chr(96 + v)); k -= v
    return ''.join(out)

for m in range(1, 6):
    letters = ALPHA[:m]
    for n in range(1, 4):
        for k in range(1, 6 * n + 2):
            brute = next((''.join(t) for t in product(letters, repeat=n) if value(t) == k), None)
            assert smallest_with_alphabet_sol(n, k, m) == brute, (n, k, m)
assert all(smallest_with_alphabet_sol(n, k, 26) == Solution().getSmallestString(n, k) for n in range(1, 8) for k in range(n, 26 * n + 1))
print("Bài 3 OK")

Bài 1: ('aabz', 'ayz')
i=0: m=5, k=100, v=max(1, 100-130)=1 -> 'a', còn k=99  | khả thi: 5 <= 99 <= 130
i=1: m=4, k= 99, v=max(1, 99-104)=1 -> 'a', còn k=98  | khả thi: 4 <= 98 <= 104
i=2: m=3, k= 98, v=max(1, 98-78)=20 -> 't', còn k=78  | khả thi: 3 <= 78 <= 78
i=3: m=2, k= 78, v=max(1, 78-52)=26 -> 'z', còn k=52  | khả thi: 2 <= 52 <= 52
i=4: m=1, k= 52, v=max(1, 52-26)=26 -> 'z', còn k=26  | khả thi: 1 <= 26 <= 26
i=5: m=0, k= 26, v=max(1, 26-0)=26 -> 'z', còn k=0  | khả thi: 0 <= 0 <= 0
=> aatzzz
Bài 3 OK


In [26]:
# Bài 4
def smallest_rtl_slicing_sol(n, k):
    q, r = divmod(k - n, 25)
    ans = ['a'] * n
    if q:
        ans[n - q:] = ['z'] * q
    if q < n:
        ans[n - q - 1] = chr(ord('a') + r)
    return ''.join(ans)

for n in range(1, 30):
    for k in range(n, 26 * n + 1):
        assert smallest_rtl_slicing_sol(n, k) == Solution().getSmallestString(n, k)
print("Bài 4 OK")

# Bài 5
def kth_smallest_string_sol(n, k, t):
    ways = [[0] * (k + 1) for _ in range(n + 1)]
    ways[0][0] = 1
    for m in range(1, n + 1):
        for s in range(1, k + 1):
            ways[m][s] = sum(ways[m - 1][s - v] for v in range(1, 27) if s - v >= 0)
    if t > ways[n][k]:
        return None
    out = []
    for i in range(n):
        m = n - i - 1
        for v in range(1, 27):
            if k - v < 0:
                break
            cnt = ways[m][k - v]
            if t <= cnt:
                out.append(chr(96 + v)); k -= v; break
            t -= cnt
    return ''.join(out)

for n in range(1, 4):
    for k in range(n, 26 * n + 1, 7):
        all_s = [''.join(tp) for tp in product(ALPHA, repeat=n) if value(tp) == k]
        for t in (1, 2, len(all_s)):
            if t <= len(all_s):
                assert kth_smallest_string_sol(n, k, t) == all_s[t - 1]
assert kth_smallest_string_sol(5, 73, 1) == Solution().getSmallestString(5, 73)
print("Bài 5 OK | xâu nhỏ thứ 1, 2, 3 với n=3, k=27:", [kth_smallest_string_sol(3, 27, t) for t in (1, 2, 3)])

Bài 4 OK
Bài 5 OK | xâu nhỏ thứ 1, 2, 3 với n=3, k=27: ['aay', 'abx', 'acw']


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "nhỏ nhất theo từ điển với tổng cố định" (giá trị mỗi vị trí trong `[lo, hi]`):**

```python
out = []
for i in range(n):
    m = n - i - 1                         # số chỗ còn lại phía sau
    v = max(lo, k - hi * m)               # nhỏ nhất mà phần sau vẫn khả thi
    out.append(to_char(v))
    k -= v
return ''.join(out)
```

**Template công thức cho bài này:** `q, r = divmod(k − n, 25)`; `'z' * n` nếu `q == n`, ngược lại `'a' * (n − q − 1) + chr(97 + r) + 'z' * q`.

**Checklist:**
- [ ] Chọn chữ **nhỏ nhất** còn khả thi (không phải lớn nhất)?
- [ ] Đổi giá trị ↔ chữ đúng (`a = 1` ⇒ `chr(96 + v)`)?
- [ ] Xử lý `k = 26n` (toàn `z`) và `k = n` (toàn `a`)?
- [ ] Không ghép chuỗi `O(n²)` trong vòng lặp?
- [ ] Kết quả có đúng độ dài `n` và giá trị `k` (tự kiểm bằng hàm `value`)?
- [ ] Chữ ký đúng: `def getSmallestString(self, n: int, k: int) -> str`?